In [4]:
# Install BERTopic and dependencies
!pip install bertopic
!pip install sentence-transformers
!pip install umap-learn
!pip install hdbscan


In [5]:
# Importing required Libraries

import pandas as pd
import numpy as np

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plt

In [6]:
# Importing cleaned/processed data

cp_df = pd.read_csv(r"C:\Users\HP\OneDrive\Documents\AMDARI\Montra Jupyter\quality_checked.csv")

In [7]:
cp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   review_id           35000 non-null  int64  
 1   booking_id          35000 non-null  int64  
 2   guest_id            35000 non-null  int64  
 3   hotel_id            35000 non-null  int64  
 4   review_text         35000 non-null  object 
 5   rating_score        35000 non-null  float64
 6   review_date         35000 non-null  object 
 7   sentiment_label     35000 non-null  object 
 8   room_type           35000 non-null  object 
 9   booking_channel     35000 non-null  object 
 10  check_in_date       35000 non-null  object 
 11  check_out_date      35000 non-null  object 
 12  total_amount        35000 non-null  float64
 13  stay_duration       35000 non-null  int64  
 14  customer_type       35000 non-null  object 
 15  loyalty_status      35000 non-null  object 
 16  coun

In [8]:
cp_df.columns

Index(['review_id', 'booking_id', 'guest_id', 'hotel_id', 'review_text',
       'rating_score', 'review_date', 'sentiment_label', 'room_type',
       'booking_channel', 'check_in_date', 'check_out_date', 'total_amount',
       'stay_duration', 'customer_type', 'loyalty_status', 'country',
       'previous_stays', 'hotel_name', 'location', 'star_rating',
       'total_rooms', 'clean_review', 'sentiment_encoded', 'word_count',
       'char_count', 'stay_duration_calc', 'price_per_night',
       'expected_sentiment', 'derived_sentiment'],
      dtype='object')

In [9]:
# Check dataframe structure
cp_df.head()

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,star_rating,total_rooms,clean_review,sentiment_encoded,word_count,char_count,stay_duration_calc,price_per_night,expected_sentiment,derived_sentiment
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,4,118,the overall price be fair for what we pay migh...,0,11,51,1,296.44,neutral,neutral
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,Deluxe,Corporate Portal,...,4,95,the room be nothing out of the ordinary for wh...,1,28,148,4,120.89,positive,positive
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,Executive,Travel Agent,...,4,231,unfortunately the pool be outdate it clearly n...,-1,16,93,2,207.80,negative,negative
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,neutral,Family Room,Airbnb,...,3,101,we be disappoint that the neighborhood be disa...,-1,21,118,3,200.39,neutral,neutral
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,Deluxe,Expedia,...,4,214,the room service be standard hotel fare for wh...,0,39,184,5,120.77,neutral,neutral


In [10]:
# Ensure no missing reviews
cp_df = cp_df.dropna(subset=['clean_review'])

# Remove exact duplicate reviews
cp_df_clean = cp_df.drop_duplicates(subset=["clean_review"])

# Convert reviews to list format
reviews = cp_df_clean['clean_review'].astype(str).tolist()

print(f"Number of reviews: {len(reviews)}")
print("Original:", len(cp_df))
print("After deduplication:", len(cp_df_clean))
reviews [:3]

Number of reviews: 34498
Original: 35000
After deduplication: 34498


['the overall price be fair for what we pay might not',
 'the room be nothing out of the ordinary for what we pay the overall price exceed our expectation a great deal and the quality easily match the price',
 'unfortunately the pool be outdate it clearly need some upkeep would not recommend it to other']

In [11]:
import re

In [12]:
from collections import Counter
import torch

In [13]:
# Normalize reviews
rev_norm = [doc.lower().strip() for doc in reviews]

# Count exact duplicates
total_rev = len(rev_norm)
unique_rev = len(set(rev_norm))
duplicated_rev = total_rev - unique_rev

print(f'Total reviews: {total_rev}')
print(f'Unique reviews: {unique_rev}')
print(f'Duplicate document: {duplicated_rev}')

if total_rev > 0:
    duplicated_rate = duplicated_rev/total_rev *100
    print(f'Duplicate rate: {duplicated_rate:.2f}%')

Total reviews: 34498
Unique reviews: 34498
Duplicate document: 0
Duplicate rate: 0.00%


In [14]:
# Calculate Vocabulary Size
all_word = " ".join(rev_norm).split()
print(f'vocab_size: {len(set(all_word))}')

# Display first 20 common words
print(Counter(all_word).most_common(20))

vocab_size: 496
[('the', 72877), ('be', 71883), ('our', 23803), ('in', 23541), ('we', 23504), ('to', 18616), ('for', 18203), ('a', 17544), ('that', 16097), ('overall', 12006), ('pay', 10935), ('stay', 10933), ('as', 10835), ('out', 10519), ('i', 10055), ('on', 10042), ('and', 9984), ('of', 9732), ('hotel', 9724), ('nothing', 9344)]


##### Generating Sentence Embeding

In [15]:
# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = embedding_model.encode(
    reviews,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1079 [00:00<?, ?it/s]

In [16]:
embeddings[:3]

array([[-0.01264704, -0.00673828,  0.02616902, ..., -0.17582549,
         0.00824867,  0.02769971],
       [ 0.0522748 , -0.0024588 ,  0.00135637, ..., -0.01036991,
        -0.08671826,  0.07271781],
       [-0.04229411,  0.03272924,  0.02205706, ..., -0.03831463,
        -0.02556697,  0.0941811 ]], shape=(3, 384), dtype=float32)

In [17]:
embeddings.shape

(34498, 384)

In [18]:
# Configure the UMAP
import umap

umap_model = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

In [19]:
# Configure the hdbscan
import hdbscan

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=800,
    min_samples=100,
    metric='euclidean',
    prediction_data=True
)

In [20]:
# Configure the CountVectorizer

from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 5),
    min_df=5
)

##### Training BERTopic Model

In [21]:
# Initialize BERTopic
topic_model = BERTopic(
    embedding_model = embedding_model,
    umap_model = umap_model,
    hdbscan_model = hdbscan_model,
    vectorizer_model = vectorizer_model,
    #language="english",
    calculate_probabilities=True,
    verbose=True
)

print(topic_model.vectorizer_model)

# Fit model
topics, probabilities = topic_model.fit_transform(
    reviews,
    embeddings
)

print(topic_model.vectorizer_model)

2026-10-20 17:19:01,993 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


CountVectorizer(min_df=5, ngram_range=(1, 5), stop_words='english')


2026-10-20 17:20:05,255 - BERTopic - Dimensionality - Completed ✓
2026-10-20 17:20:05,258 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-20 17:20:13,673 - BERTopic - Cluster - Completed ✓
2026-10-20 17:20:13,682 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-20 17:20:16,127 - BERTopic - Representation - Completed ✓


CountVectorizer(min_df=5, ngram_range=(1, 5), stop_words='english')


In [22]:
topic_model.reduce_topics(reviews, nr_topics = 'auto')

2026-10-20 17:20:16,555 - BERTopic - Topic reduction - Reducing number of topics
2026-10-20 17:20:16,576 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-20 17:20:19,300 - BERTopic - Representation - Completed ✓
2026-10-20 17:20:19,308 - BERTopic - Topic reduction - Reduced number of topics from 13 to 13


##### Add Topic information to data frame

In [23]:
# Assign topic labels
cp_df_clean['Topic'] = topics

# Optional: assign topic probability
cp_df_clean['Topic_Probability'] = probabilities.max(axis=1).round(2)

cp_df_clean.head()

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,room_type,booking_channel,...,clean_review,sentiment_encoded,word_count,char_count,stay_duration_calc,price_per_night,expected_sentiment,derived_sentiment,Topic,Topic_Probability
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,Suite,Direct Website,...,the overall price be fair for what we pay migh...,0,11,51,1,296.44,neutral,neutral,0,0.49
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,Deluxe,Corporate Portal,...,the room be nothing out of the ordinary for wh...,1,28,148,4,120.89,positive,positive,0,0.39
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,Executive,Travel Agent,...,unfortunately the pool be outdate it clearly n...,-1,16,93,2,207.80,negative,negative,6,1.00
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,neutral,Family Room,Airbnb,...,we be disappoint that the neighborhood be disa...,-1,21,118,3,200.39,neutral,neutral,0,0.83
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,Deluxe,Expedia,...,the room service be standard hotel fare for wh...,0,39,184,5,120.77,neutral,neutral,0,0.11


##### View discovered topics

In [24]:
# Topic summary
topic_info = topic_model.get_topic_info()

topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1661,-1_spa_receptionist_overall_pay,"[spa, receptionist, overall, pay, clearly, ite...",[the restaurant leave a lot to be desire sever...
1,0,9403,0_overall_pay_check_hotel,"[overall, pay, check, hotel, staff, arrival, s...",[overall our arrival experience be acceptable ...
2,1,9089,1_time_hotel_stay_fine,"[time, hotel, stay, fine, glasgow, visit, over...",[this hotel be a fine if unremarkable choice f...
3,2,3586,2_buffet_option_cafe_house cafe,"[buffet, option, cafe, house cafe, house, dini...",[arrival be unremarkable I would describe the ...
4,3,1950,3_gym_fitness_fitness facility_facility,"[gym, fitness, fitness facility, facility, pay...","[the fitness facility be standard, the fitness..."
5,4,1596,4_belfast_stay belfast_visit belfast_hotel,"[belfast, stay belfast, visit belfast, hotel, ...",[this hotel be a fine if unremarkable choice f...
6,5,1419,5_bathroom_bathroom area_area_clean,"[bathroom, bathroom area, area, clean, leave, ...",[sadly the pricing turn out to be not worth wh...
7,6,1147,6_pool_overall pool_stay_overall,"[pool, overall pool, stay, overall, pay, fine,...",[the rate we pay be also the pool be basic but...
8,7,1081,7_fi_wi_wi fi_overall wi fi,"[fi, wi, wi fi, overall wi fi, overall wi, sta...",[the neighborhood be fine on top of that the w...
9,8,984,8_base_ask base_base visit_ask base visit,"[base, ask base, base visit, ask base visit, a...",[we could not have ask for a well base while v...


##### Examine Keywords for each topic

In [25]:
# Display top words for each topic
for topic in topic_info['Topic'][:10]:
    
    if topic != -1:
        print(f"\nTopic {topic}")
        print(topic_model.get_topic(topic))


Topic 0
[('overall', np.float64(0.025465436528728937)), ('pay', np.float64(0.022480419992551553)), ('check', np.float64(0.017449504216692596)), ('hotel', np.float64(0.017047733166089856)), ('staff', np.float64(0.015997281214049322)), ('arrival', np.float64(0.015494139176688857)), ('stay', np.float64(0.014705475262775585)), ('price', np.float64(0.01443339407245609)), ('desk', np.float64(0.014094934802053453)), ('area', np.float64(0.013077241097545341))]

Topic 1
[('time', np.float64(0.03083297043552961)), ('hotel', np.float64(0.029127633808733806)), ('stay', np.float64(0.02750613989411188)), ('fine', np.float64(0.020251282422249085)), ('glasgow', np.float64(0.018585831443820054)), ('visit', np.float64(0.018192686661391278)), ('overall', np.float64(0.017692964394137374)), ('pay', np.float64(0.0169575623740117)), ('unremarkable', np.float64(0.01651572232870435)), ('stand', np.float64(0.016156856300037967))]

Topic 2
[('buffet', np.float64(0.025182466967611634)), ('option', np.float64(0.0

##### Retrieving Representative Reviews Per Topic

In [26]:
representative_docs = topic_model.get_representative_docs()

for topic, docs in representative_docs.items():
    
    print(f"\n{'='*60}")
    print(f"TOPIC {topic}")
    
    for doc in docs[:3]:
        print(f"- {doc}")


TOPIC -1
- the restaurant leave a lot to be desire several item be clearly reheat in addition we be disappoint that the check in desk be chaotic
- we be disappoint that the manager be dismissive that say the room service leave a lot to be desire several item be clearly reheat
- the dining option leave a lot to be desire several item be clearly reheat leave feel pretty disappointed overall

TOPIC 0
- overall our arrival experience be acceptable also overall the neighborhood be not the most convenient but manageable meanwhile the front desk team be okay overall
- arrival leave a lot to be desire our room be not ready despite an on time arrival meanwhile the overall price leave a lot to be desire several amenity cost extra on top of the rate leave feel pretty disappointed overall
- overall our arrival experience be about average also I would describe the rate we pay as in line with similar hotel it match what similar hotel charge

TOPIC 1
- this hotel be a fine if unremarkable choice for

In [27]:
topic_model.visualize_barchart(top_n_topics=12)

In [28]:
topic_model.visualize_topics()

In [29]:
topic_model.visualize_hierarchy()

In [30]:
topic_model.visualize_heatmap()

In [31]:
custom_labels = topic_model.generate_topic_labels()

topic_model.set_topic_labels(custom_labels)

topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,1661,-1_spa_receptionist_overall_pay,-1_spa_receptionist_overall,"[spa, receptionist, overall, pay, clearly, ite...",[the restaurant leave a lot to be desire sever...
1,0,9403,0_overall_pay_check_hotel,0_overall_pay_check,"[overall, pay, check, hotel, staff, arrival, s...",[overall our arrival experience be acceptable ...
2,1,9089,1_time_hotel_stay_fine,1_time_hotel_stay,"[time, hotel, stay, fine, glasgow, visit, over...",[this hotel be a fine if unremarkable choice f...
3,2,3586,2_buffet_option_cafe_house cafe,2_buffet_option_cafe,"[buffet, option, cafe, house cafe, house, dini...",[arrival be unremarkable I would describe the ...
4,3,1950,3_gym_fitness_fitness facility_facility,3_gym_fitness_fitness facility,"[gym, fitness, fitness facility, facility, pay...","[the fitness facility be standard, the fitness..."
5,4,1596,4_belfast_stay belfast_visit belfast_hotel,4_belfast_stay belfast_visit belfast,"[belfast, stay belfast, visit belfast, hotel, ...",[this hotel be a fine if unremarkable choice f...
6,5,1419,5_bathroom_bathroom area_area_clean,5_bathroom_bathroom area_area,"[bathroom, bathroom area, area, clean, leave, ...",[sadly the pricing turn out to be not worth wh...
7,6,1147,6_pool_overall pool_stay_overall,6_pool_overall pool_stay,"[pool, overall pool, stay, overall, pay, fine,...",[the rate we pay be also the pool be basic but...
8,7,1081,7_fi_wi_wi fi_overall wi fi,7_fi_wi_wi fi,"[fi, wi, wi fi, overall wi fi, overall wi, sta...",[the neighborhood be fine on top of that the w...
9,8,984,8_base_ask base_base visit_ask base visit,8_base_ask base_base visit,"[base, ask base, base visit, ask base visit, a...",[we could not have ask for a well base while v...


In [32]:
topic_df= cp_df_clean

In [33]:
# Save topic assignments

topic_df = cp_df_clean
topic_df.to_csv(
    'reviews_with_topics.csv',
    index=False
)

# Save topic summary
topic_info.to_csv(
    'topic_summary.csv',
    index=False
)

In [34]:
topic_distribution = (
    topic_df['Topic']
    .value_counts(normalize=True)
    .reset_index()
)

topic_distribution.columns = [
    'Topic',
    'Percentage'
]

topic_distribution['Percentage'] *= 100

topic_distribution.sort_values(
    by='Percentage',
    ascending=False,
    inplace=True
)

topic_distribution.head(10)

,Topic,Percentage
0,0,27.256653
1,1,26.346455
2,2,10.394805
3,3,5.652502
4,-1,4.814772
5,4,4.626355
6,5,4.113282
7,6,3.324830
8,7,3.133515
9,8,2.852339


In [35]:
top_topics = topic_model.get_topic_info().head(10)

print(top_topics[['Topic', 'Count', 'Name']])

   Topic  Count                                        Name
0     -1   1661             -1_spa_receptionist_overall_pay
1      0   9403                   0_overall_pay_check_hotel
2      1   9089                      1_time_hotel_stay_fine
3      2   3586             2_buffet_option_cafe_house cafe
4      3   1950     3_gym_fitness_fitness facility_facility
5      4   1596  4_belfast_stay belfast_visit belfast_hotel
6      5   1419         5_bathroom_bathroom area_area_clean
7      6   1147            6_pool_overall pool_stay_overall
8      7   1081                 7_fi_wi_wi fi_overall wi fi
9      8    984   8_base_ask base_base visit_ask base visit
